In [ ]:
!pip install praw vaderSentiment wordcloud pyLDAvis gensim --quiet

import praw
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from wordcloud import WordCloud
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from collections import Counter
import nltk
import re
import warnings
warnings.filterwarnings("ignore")

nltk.download("stopwords")
nltk.download("punkt")
from nltk.corpus import stopwords

In [ ]:
import time
from datetime import datetime

CLIENT_ID     = ""
CLIENT_SECRET = ""
USER_AGENT    = "elon_musk_analysis/1.0"

reddit = praw.Reddit(
    client_id     = CLIENT_ID,
    client_secret = CLIENT_SECRET,
    user_agent    = USER_AGENT
)

SUBREDDITS = ["elonmusk", "tesla", "spacex", "technology", "worldnews"]
LIMIT      = 150

posts = []

for sub_name in SUBREDDITS:
    print(f"Scanning r/{sub_name}...")
    try:
        subreddit = reddit.subreddit(sub_name)
        for post in subreddit.search("Elon Musk", limit=LIMIT, sort="new"):
            posts.append({
                "id"           : post.id,
                "subreddit"    : sub_name,
                "title"        : post.title,
                "selftext"     : post.selftext,
                "score"        : post.score,
                "upvote_ratio" : post.upvote_ratio,
                "num_comments" : post.num_comments,
                "author"       : str(post.author),
                "created_utc"  : datetime.utcfromtimestamp(post.created_utc),
                "url"          : post.url,
                "flair"        : post.link_flair_text
            })
        time.sleep(2)
    except Exception as e:
        print(f"  Error: {e}")

df = pd.DataFrame(posts)
df.drop_duplicates(subset="id", inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"\nTotal posts collected: {len(df)}")
print(f"Date range: {df['created_utc'].min().date()} to {df['created_utc'].max().date()}")
print(f"\nSubreddit distribution:")
print(df['subreddit'].value_counts())

In [ ]:
import re
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

print("RAW DATA SAMPLE (first 3 rows):")
print(df[["subreddit", "title", "score", "num_comments", "created_utc"]].head(3))

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words and len(w) > 2]
    return " ".join(tokens)

df["clean_title"]    = df["title"].apply(clean_text)
df["clean_selftext"] = df["selftext"].apply(clean_text)
df["full_text"]      = df["clean_title"] + " " + df["clean_selftext"]
df["full_text"]      = df["full_text"].str.strip()

df = df[df["full_text"].str.len() > 10].copy()
df.reset_index(drop=True, inplace=True)

df["date"]  = pd.to_datetime(df["created_utc"]).dt.date
df["year"]  = pd.to_datetime(df["created_utc"]).dt.year
df["month"] = pd.to_datetime(df["created_utc"]).dt.to_period("M")

print(f"\nAfter cleaning: {len(df)} posts remain")
print("\nCLEANED DATA SAMPLE (first 3 rows):")
print(df[["subreddit", "clean_title", "clean_selftext", "year"]].head(3))

print("\nSUMMARY STATISTICS:")
print(df[["score", "upvote_ratio", "num_comments"]].describe().round(2))
print(f"\nPosts with empty 'selftext' (title-only): {(df['selftext']=='').sum()}")

In [ ]:
from collections import Counter

sns.set_style("whitegrid")
extra_stop = {"elon", "musk", "elonmusk", "said", "says", "also",
              "one", "new", "would", "could", "like", "get",
              "use", "make", "just", "will", "two", "may",
              "even", "much", "many", "still", "thats", "dont",
              "what", "when", "where", "this", "that", "with",
              "from", "have", "they", "their", "your", "about"}

plt.figure(figsize=(10, 6))
sub_counts = df["subreddit"].value_counts()
bars = plt.bar(sub_counts.index, sub_counts.values,
               color=["#FF4500","#0079D3","#46D160","#FFD635","#FF585B"],
               edgecolor="black")
plt.title("Figure 1: Number of Posts about Elon Musk per Subreddit",
          fontweight="bold", fontsize=13)
plt.xlabel("Subreddit")
plt.ylabel("Post Count")
for bar, v in zip(bars, sub_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, v + 2, str(v),
             ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig("fig1_subreddit_count.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(11, 6))
yearly = df.groupby("year").size().reset_index(name="count")
plt.plot(yearly["year"], yearly["count"], marker="o",
         color="#FF4500", linewidth=2.5, markersize=10)
plt.fill_between(yearly["year"], yearly["count"], alpha=0.25, color="#FF4500")
plt.title("Figure 2: Yearly Volume of Elon Musk Discussions on Reddit",
          fontweight="bold", fontsize=13)
plt.xlabel("Year")
plt.ylabel("Post Count")
plt.xticks(yearly["year"], rotation=45)
for x, y in zip(yearly["year"], yearly["count"]):
    plt.text(x, y + 5, str(y), ha="center", fontweight="bold", fontsize=9)
plt.tight_layout()
plt.savefig("fig2_yearly_trend.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(11, 6))
sub_order = df.groupby("subreddit")["score"].median().sort_values(ascending=False).index
data_box = [df[df["subreddit"]==s]["score"].values for s in sub_order]
bp = plt.boxplot(data_box, labels=sub_order, patch_artist=True, showfliers=False)
colors = ["#FF4500","#0079D3","#46D160","#FFD635","#FF585B"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
plt.title("Figure 3: Post Score Distribution per Subreddit (Engagement)",
          fontweight="bold", fontsize=13)
plt.xlabel("Subreddit")
plt.ylabel("Score (Outliers Excluded)")
plt.tight_layout()
plt.savefig("fig3_score_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(11, 7))
scatter_df = df[df["score"] < 20000]
sc = plt.scatter(scatter_df["score"], scatter_df["num_comments"],
                 c=scatter_df["upvote_ratio"], cmap="RdYlGn",
                 alpha=0.7, s=60, edgecolor="black", linewidth=0.3)
cbar = plt.colorbar(sc, label="Upvote Ratio (Positive Reception)")
plt.title("Figure 4: Post Score vs Number of Comments (Color = Upvote Ratio)",
          fontweight="bold", fontsize=13)
plt.xlabel("Score (Net Upvotes)")
plt.ylabel("Number of Comments")
plt.tight_layout()
plt.savefig("fig4_score_vs_comments.png", dpi=150, bbox_inches="tight")
plt.show()

all_words = " ".join(df["full_text"]).split()
all_words = [w for w in all_words if w not in extra_stop and len(w) > 3]
word_freq  = Counter(all_words).most_common(20)
words, freqs = zip(*word_freq)

plt.figure(figsize=(10, 8))
plt.barh(words[::-1], freqs[::-1], color="#0079D3", edgecolor="black")
plt.title("Figure 5: Top 20 Most Frequent Words in Elon Musk Discussions",
          fontweight="bold", fontsize=13)
plt.xlabel("Frequency")
plt.ylabel("Word")
for i, v in enumerate(freqs[::-1]):
    plt.text(v + 1, i, str(v), va="center", fontsize=9)
plt.tight_layout()
plt.savefig("fig5_top_words.png", dpi=150, bbox_inches="tight")
plt.show()


text_all = " ".join(df["full_text"])
wc = WordCloud(width=1200, height=600, background_color="white",
               colormap="Reds", max_words=100,
               stopwords=extra_stop, collocations=False).generate(text_all)

plt.figure(figsize=(14, 7))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title(" Word Cloud of Elon Musk Discussions",
          fontweight="bold", fontsize=14, pad=15)
plt.tight_layout()
plt.savefig("wordcloud.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nTOP 10 HIGHEST SCORING POSTS:")
top10 = df[["subreddit","title","score","num_comments"]].sort_values(
    "score", ascending=False).head(10)
print(top10.to_string(index=False))

print("\nAll exploratory visualisations created successfully!")

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import numpy as np

analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return 0.0
    return analyzer.polarity_scores(text)["compound"]

def label_sentiment(score):
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"

df["sentiment_score"] = df["title"].apply(get_sentiment)
df["sentiment_label"] = df["sentiment_score"].apply(label_sentiment)

print("Sentiment Analysis Summary:")
print(df["sentiment_label"].value_counts())
print(f"\nAverage sentiment score: {df['sentiment_score'].mean():.3f}")

plt.figure(figsize=(9, 6))
sentiment_counts = df["sentiment_label"].value_counts()
colors_sent = {"Positive": "#46D160", "Neutral": "#A0A0A0", "Negative": "#FF4500"}
bar_colors = [colors_sent[s] for s in sentiment_counts.index]
bars = plt.bar(sentiment_counts.index, sentiment_counts.values,
               color=bar_colors, edgecolor="black")
plt.title("Figure 6: Overall Sentiment Distribution of Elon Musk Posts (VADER)",
          fontweight="bold", fontsize=13)
plt.xlabel("Sentiment Category")
plt.ylabel("Post Count")
total = sentiment_counts.sum()
for bar, v in zip(bars, sentiment_counts.values):
    pct = v / total * 100
    plt.text(bar.get_x() + bar.get_width()/2, v + 5,
             f"{v}\n({pct:.1f}%)", ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig("fig6_sentiment_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


plt.figure(figsize=(11, 6))
sent_by_sub = df.groupby(["subreddit", "sentiment_label"]).size().unstack(fill_value=0)
sent_by_sub_pct = sent_by_sub.div(sent_by_sub.sum(axis=1), axis=0) * 100
sent_by_sub_pct = sent_by_sub_pct[["Positive", "Neutral", "Negative"]]
sent_by_sub_pct.plot(kind="bar", stacked=True,
                     color=["#46D160", "#A0A0A0", "#FF4500"],
                     edgecolor="black", ax=plt.gca())
plt.title("Figure 7: Sentiment Composition per Subreddit (%)",
          fontweight="bold", fontsize=13)
plt.xlabel("Subreddit")
plt.ylabel("Percentage (%)")
plt.legend(title="Sentiment", loc="lower right")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("fig7_sentiment_by_subreddit.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(12, 6))
yearly_sent = df.groupby("year")["sentiment_score"].mean().reset_index()
colors_year = ["#46D160" if v > 0 else "#FF4500" for v in yearly_sent["sentiment_score"]]
plt.bar(yearly_sent["year"], yearly_sent["sentiment_score"],
        color=colors_year, edgecolor="black")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Figure 8: Average Sentiment Score Trend by Year",
          fontweight="bold", fontsize=13)
plt.xlabel("Year")
plt.ylabel("Mean VADER Compound Score")
plt.xticks(yearly_sent["year"], rotation=45)
for x, y in zip(yearly_sent["year"], yearly_sent["sentiment_score"]):
    plt.text(x, y + (0.01 if y >= 0 else -0.02), f"{y:.2f}",
             ha="center", fontweight="bold", fontsize=9)
plt.tight_layout()
plt.savefig("fig8_sentiment_trend.png", dpi=150, bbox_inches="tight")
plt.show()

vectorizer = CountVectorizer(ngram_range=(2, 2), stop_words="english",
                             max_features=20)
bigrams = vectorizer.fit_transform(df["full_text"])
bigram_freq = bigrams.sum(axis=0).A1
bigram_names = vectorizer.get_feature_names_out()
bigram_df = pd.DataFrame({"bigram": bigram_names, "freq": bigram_freq})
bigram_df = bigram_df.sort_values("freq", ascending=True)

plt.figure(figsize=(11, 8))
plt.barh(bigram_df["bigram"], bigram_df["freq"],
         color="#FF4500", edgecolor="black")
plt.title("Figure 9: Top 20 Most Frequent Bigrams (Two-word Phrases)",
          fontweight="bold", fontsize=13)
plt.xlabel("Frequency")
plt.ylabel("Bigram")
for i, v in enumerate(bigram_df["freq"]):
    plt.text(v + 1, i, str(v), va="center", fontsize=9)
plt.tight_layout()
plt.savefig("fig9_top_bigrams.png", dpi=150, bbox_inches="tight")
plt.show()

# Figure 10 - LDA Topic Modeling (5 Topics)
vectorizer_lda = CountVectorizer(stop_words="english", max_features=1000,
                                  min_df=5, max_df=0.7)
dtm = vectorizer_lda.fit_transform(df["full_text"])

n_topics = 5
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42,
                                learning_method="online", max_iter=20)
lda.fit(dtm)

feature_names = vectorizer_lda.get_feature_names_out()
top_n = 10

fig, axes = plt.subplots(1, n_topics, figsize=(22, 6), sharex=False)
fig.suptitle("Figure 10: LDA Topic Modeling — Top 10 Words per Topic",
             fontweight="bold", fontsize=14, y=1.02)

topic_colors = ["#FF4500", "#0079D3", "#46D160", "#FFD635", "#9B59B6"]

for topic_idx, topic in enumerate(lda.components_):
    top_idx = topic.argsort()[-top_n:]
    top_words = [feature_names[i] for i in top_idx]
    top_weights = topic[top_idx]
    axes[topic_idx].barh(top_words, top_weights,
                          color=topic_colors[topic_idx], edgecolor="black")
    axes[topic_idx].set_title(f"Topic {topic_idx + 1}", fontweight="bold")
    axes[topic_idx].set_xlabel("Weight")
plt.tight_layout()
plt.savefig("fig10_lda_topics.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nLDA TOPIC SUMMARY (Top 10 words per topic):")
for topic_idx, topic in enumerate(lda.components_):
    top_idx = topic.argsort()[-10:][::-1]
    top_words = [feature_names[i] for i in top_idx]
    print(f"  Topic {topic_idx + 1}: {', '.join(top_words)}")

print("\nTEXT MINING RESULTS:")
print(f"Positive posts: {(df['sentiment_label']=='Positive').sum()} "
      f"({(df['sentiment_label']=='Positive').mean()*100:.1f}%)")
print(f"Negative posts: {(df['sentiment_label']=='Negative').sum()} "
      f"({(df['sentiment_label']=='Negative').mean()*100:.1f}%)")
print(f"Neutral posts:  {(df['sentiment_label']=='Neutral').sum()} "
      f"({(df['sentiment_label']=='Neutral').mean()*100:.1f}%)")

print("\nMost negative posts (top 5):")
print(df.nsmallest(5, "sentiment_score")[["subreddit","title","sentiment_score"]].to_string(index=False))

print("\nMost positive posts (top 5):")
print(df.nlargest(5, "sentiment_score")[["subreddit","title","sentiment_score"]].to_string(index=False))